In [1]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

In [2]:
pwd

'/Users/minhtamninale/Documents/Angela/Spots_troubleshooting/3rd_647_as_PPE/Post_processing'

In [4]:
df = pd.DataFrame()

folder_path = '../Late_embryo'

dfs = []

for filename in os.listdir(folder_path):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path, sep ='\s+', header=None, dtype={3: str}, names=['x', 'y', 'z', 'conformation'])
        dfs.append(df)
        
df = pd.concat(dfs, ignore_index=True)

file_paths =glob.glob('../Late_embryo/*.txt')

for file_path in file_paths:
    file_data = pd.read_csv(file_path, sep='\s+', header=None, dtype={3: str}, names=['x', 'y', 'z', 'conformation'])
    
      
    file_name = file_path.split('/')[-1]
    number = file_name.split('_')[0]
    number = int(number)
    
    file_data['Embryo_ID'] = number
    
    df = pd.concat([df,file_data],ignore_index=True)
    
    main_frame = df.dropna()

In [5]:
main_frame

,x,y,z,conformation,Embryo_ID
1079,43.608781,1.254454,5.76,111,18.0
1080,85.368893,3.136135,5.28,111,18.0
1081,84.576606,18.981869,6.60,111,18.0
1082,5.810103,80.549148,6.84,111,18.0
1083,0.066024,49.220812,6.00,000,18.0
...,...,...,...,...,...
2153,72.131102,46.084677,5.04,010,38.0
2154,67.014251,58.299097,5.04,010,38.0
2155,12.214420,27.895094,1.08,000,38.0
2156,56.120308,31.493397,4.08,000,38.0


In [6]:
def conformation_meaning(conformation): 
    meanings = { 
        '000': 'Not_touching',
        '100': '3_prime_touch_PPE',
        '010':'5_prime_touch_PPE',
        '001':'3_prime_touch_5_prime',
        '110':'PPE_prime_middle',
        '011':'5_middle',
        '101':'3_prime_middle',
        '111':'All_touching'
    }
    if conformation in meanings: 
        return meanings[conformation]
    else:
        return 'Not supported'
    
main_frame.loc[:,'Meaning'] = main_frame['conformation'].apply(conformation_meaning)

/var/folders/6g/g2plc5bd3jv_sqpbk23jqwm40000gn/T/ipykernel_14249/1598569033.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_frame.loc[:,'Meaning'] = main_frame['conformation'].apply(conformation_meaning)


In [7]:
def count_conformation_per_embryo(df):
    counts = {}
    embryo_ids = main_frame['Embryo_ID'].unique()
    for embryo_id in embryo_ids:
        embryo_df = main_frame.loc[main_frame['Embryo_ID'] == embryo_id]
        count_000 = len(embryo_df.loc[embryo_df["conformation"] == '000'])
        count_100 = len(embryo_df.loc[embryo_df["conformation"] == '100'])
        count_010 = len(embryo_df.loc[embryo_df["conformation"] == '010'])
        count_001 = len(embryo_df.loc[embryo_df["conformation"] == '001'])
        count_110 = len(embryo_df.loc[embryo_df["conformation"] == '110'])
        count_011 = len(embryo_df.loc[embryo_df["conformation"] == '011'])
        count_101 = len(embryo_df.loc[embryo_df["conformation"] == '101'])
        count_111 = len(embryo_df.loc[embryo_df["conformation"] == '111'])
        
        counts[embryo_id] = {
            'Counts_of_not_touching':count_000, 
            'Counts_3_prime_touch_PPE': count_100,
            'Counts_5_prime_touch_PPE': count_010,
            'Counts_3_prime_touch_5_prime':count_001,
            'Counts_PPE_prime_middle:':count_110,
            'Counts_5_middle':count_011,
            'Counts_3_prime_middle':count_101,
            'Counts_all_touching':count_111
        }
        
    counts_df = pd.DataFrame(counts).T.reset_index()
    counts_df.columns = ["Embryo_ID", "Counts_of_not_touching","Counts_3_prime_touch_5_prime", 
                        "Counts_5_prime_touch_PPE","Counts_3_prime_touch_PPE","Counts_5_prime_middle",
                        "Counts_PPE_middle","Counts_3_prime_middle","Counts_all_touching"]
    
    return counts_df

In [8]:
counts_conformation = count_conformation_per_embryo(main_frame).sort_values("Embryo_ID")
counts_conformation

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
2,9.0,3,2,10,4,2,1,0,23
9,10.0,8,10,20,24,2,4,0,50
4,15.0,2,2,9,4,0,3,0,12
12,16.0,21,9,33,29,4,7,3,64
13,17.0,1,2,9,1,0,5,0,33
0,18.0,2,1,0,3,0,0,1,10
10,19.0,6,2,12,8,0,1,1,23
3,23.0,1,2,14,10,0,1,2,16
11,24.0,2,4,15,17,2,1,0,41
1,30.0,0,4,4,4,0,1,0,30


In [9]:
df_2 = pd.read_csv('../../2nd_lamin_test/Conformation_results/Late_conformation_conformation_count.csv')
df_2_new = df_2.drop(columns=['Unnamed: 0'])

In [10]:
concatenated_df = pd.concat([df_2_new, counts_conformation])
concatenated_df

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
0,10.0,19,0,7,1,0,0,0,0
1,19.0,8,0,6,0,1,0,1,0
2,11.0,61,4,6,0,0,0,0,0
3,9.0,19,2,7,1,0,0,0,1
4,13.0,40,0,4,1,0,0,0,0
5,17.0,24,3,7,6,0,0,0,1
6,6.0,24,1,8,3,0,0,0,2
7,2.0,8,1,12,1,0,0,0,3
8,8.0,30,1,3,0,0,0,0,1
9,20.0,0,1,1,0,0,0,0,0


In [11]:
counts_conformation.to_csv('../Late_embryo/late_conformation_conformation_count.csv')

In [13]:
df_2 = pd.read_csv('../../2nd_lamin_test/Conformation_results/Late_conformation_conformation_count.csv')
df_2_new = df_2.drop(columns=['Unnamed: 0'])
df_2_new

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
0,10.0,19,0,7,1,0,0,0,0
1,19.0,8,0,6,0,1,0,1,0
2,11.0,61,4,6,0,0,0,0,0
3,9.0,19,2,7,1,0,0,0,1
4,13.0,40,0,4,1,0,0,0,0
5,17.0,24,3,7,6,0,0,0,1
6,6.0,24,1,8,3,0,0,0,2
7,2.0,8,1,12,1,0,0,0,3
8,8.0,30,1,3,0,0,0,0,1
9,20.0,0,1,1,0,0,0,0,0


In [14]:
concatenated_df = pd.concat([df_2_new, counts_conformation])
concatenated_df

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
0,10.0,19,0,7,1,0,0,0,0
1,19.0,8,0,6,0,1,0,1,0
2,11.0,61,4,6,0,0,0,0,0
3,9.0,19,2,7,1,0,0,0,1
4,13.0,40,0,4,1,0,0,0,0
5,17.0,24,3,7,6,0,0,0,1
6,6.0,24,1,8,3,0,0,0,2
7,2.0,8,1,12,1,0,0,0,3
8,8.0,30,1,3,0,0,0,0,1
9,20.0,0,1,1,0,0,0,0,0


In [15]:
def calculate_category_percentages(dataframe):
    #Exclude Embryo ID in calculation
    columns_to_calculate = dataframe.columns[1:]
    
    # Calculate the sum of each row
    row_sums = dataframe[columns_to_calculate].sum(axis=1)

    # Calculate the percentage of each column for each row by dividing the value of each column by the row sum and multiplying by 100
    category_percentages = dataframe[columns_to_calculate].div(row_sums, axis=0) *100
    
    category_percentages.insert(0, 'Embryo_ID', dataframe['Embryo_ID'])
    return category_percentages

In [16]:
percentage_df = calculate_category_percentages(concatenated_df)
percentage_df

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
0,10.0,70.370370,0.000000,25.925926,3.703704,0.000000,0.000000,0.000000,0.000000
1,19.0,50.000000,0.000000,37.500000,0.000000,6.250000,0.000000,6.250000,0.000000
2,11.0,85.915493,5.633803,8.450704,0.000000,0.000000,0.000000,0.000000,0.000000
3,9.0,63.333333,6.666667,23.333333,3.333333,0.000000,0.000000,0.000000,3.333333
4,13.0,88.888889,0.000000,8.888889,2.222222,0.000000,0.000000,0.000000,0.000000
5,17.0,58.536585,7.317073,17.073171,14.634146,0.000000,0.000000,0.000000,2.439024
6,6.0,63.157895,2.631579,21.052632,7.894737,0.000000,0.000000,0.000000,5.263158
7,2.0,32.000000,4.000000,48.000000,4.000000,0.000000,0.000000,0.000000,12.000000
8,8.0,85.714286,2.857143,8.571429,0.000000,0.000000,0.000000,0.000000,2.857143
9,20.0,0.000000,50.000000,50.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [17]:
transposed_percentage= percentage_df.T
transposed_percentage

,0,1,2,3,4,5,6,7,8,9,...,0,10,3,11,1,5,8,14,6,7
Embryo_ID,10.000000,19.00,11.000000,9.000000,13.000000,17.000000,6.000000,2.0,8.000000,20.0,...,18.000000,19.000000,23.000000,24.000000,30.000000,31.0,36.000000,38.000000,39.000000,42.000000
Counts_of_not_touching,70.370370,50.00,85.915493,63.333333,88.888889,58.536585,63.157895,32.0,85.714286,0.0,...,11.764706,11.320755,2.173913,2.439024,0.000000,12.5,6.962025,15.584416,7.142857,12.222222
Counts_3_prime_touch_5_prime,0.000000,0.00,5.633803,6.666667,0.000000,7.317073,2.631579,4.0,2.857143,50.0,...,5.882353,3.773585,4.347826,4.878049,9.302326,12.5,8.860759,12.987013,14.285714,7.777778
Counts_5_prime_touch_PPE,25.925926,37.50,8.450704,23.333333,8.888889,17.073171,21.052632,48.0,8.571429,50.0,...,0.000000,22.641509,30.434783,18.292683,9.302326,2.5,17.721519,25.974026,25.000000,17.777778
Counts_3_prime_touch_PPE,3.703704,0.00,0.000000,3.333333,2.222222,14.634146,7.894737,4.0,0.000000,0.0,...,17.647059,15.094340,21.739130,20.731707,9.302326,7.5,17.721519,22.077922,10.714286,20.000000
Counts_5_prime_middle,0.000000,6.25,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.000000,0.000000,0.000000,2.439024,0.000000,12.5,4.430380,2.597403,0.000000,2.222222
Counts_PPE_middle,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,0.000000,1.886792,2.173913,1.219512,2.325581,2.5,3.797468,0.000000,3.571429,5.555556
Counts_3_prime_middle,0.000000,6.25,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,...,5.882353,1.886792,4.347826,0.000000,0.000000,2.5,1.898734,0.000000,0.000000,2.222222
Counts_all_touching,0.000000,0.00,0.000000,3.333333,0.000000,2.439024,5.263158,12.0,2.857143,0.0,...,58.823529,43.396226,34.782609,50.000000,69.767442,47.5,38.607595,20.779221,39.285714,32.222222


In [19]:
percentage_df.to_csv('../Late_embryo/Late_percentages_per_embryo.csv')

In [20]:
transposed_percentage.to_csv('../Late_embryo/Late_percentages.csv')

In [21]:
transposed_df=concatenated_df.T
transposed_df

,0,1,2,3,4,5,6,7,8,9,...,0,10,3,11,1,5,8,14,6,7
Embryo_ID,10.0,19.0,11.0,9.0,13.0,17.0,6.0,2.0,8.0,20.0,...,18.0,19.0,23.0,24.0,30.0,31.0,36.0,38.0,39.0,42.0
Counts_of_not_touching,19.0,8.0,61.0,19.0,40.0,24.0,24.0,8.0,30.0,0.0,...,2.0,6.0,1.0,2.0,0.0,5.0,11.0,12.0,2.0,11.0
Counts_3_prime_touch_5_prime,0.0,0.0,4.0,2.0,0.0,3.0,1.0,1.0,1.0,1.0,...,1.0,2.0,2.0,4.0,4.0,5.0,14.0,10.0,4.0,7.0
Counts_5_prime_touch_PPE,7.0,6.0,6.0,7.0,4.0,7.0,8.0,12.0,3.0,1.0,...,0.0,12.0,14.0,15.0,4.0,1.0,28.0,20.0,7.0,16.0
Counts_3_prime_touch_PPE,1.0,0.0,0.0,1.0,1.0,6.0,3.0,1.0,0.0,0.0,...,3.0,8.0,10.0,17.0,4.0,3.0,28.0,17.0,3.0,18.0
Counts_5_prime_middle,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,0.0,5.0,7.0,2.0,0.0,2.0
Counts_PPE_middle,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,1.0,6.0,0.0,1.0,5.0
Counts_3_prime_middle,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2.0,0.0,0.0,1.0,3.0,0.0,0.0,2.0
Counts_all_touching,0.0,0.0,0.0,1.0,0.0,1.0,2.0,3.0,1.0,0.0,...,10.0,23.0,16.0,41.0,30.0,19.0,61.0,16.0,11.0,29.0


In [22]:
transposed_df.shape

(9, 25)

In [23]:
def calculate_category_sum(dataframe): 
    categories = ['Embryo_ID','Counts_of_not_touching', 'Counts_3_prime_touch_5_prime', 'Counts_5_prime_touch_PPE',
                  'Counts_3_prime_touch_PPE', 'Counts_5_prime_middle', 'Counts_PPE_middle',
                  'Counts_3_prime_middle', 'Counts_all_touching']    
    row_sums = []
    for _, row in dataframe.iterrows():
        row_sum = row.sum()
        row_sums.append(row_sum)
    row_sums_df = pd.DataFrame({'Category': categories, 'Category_Sum':row_sums})
    return row_sums_df

In [24]:
late_sum = calculate_category_sum(transposed_df).iloc[1:]
late_sum

,Category,Category_Sum
1,Counts_of_not_touching,320.0
2,Counts_3_prime_touch_5_prime,91.0
3,Counts_5_prime_touch_PPE,259.0
4,Counts_3_prime_touch_PPE,186.0
5,Counts_5_prime_middle,27.0
6,Counts_PPE_middle,37.0
7,Counts_3_prime_middle,14.0
8,Counts_all_touching,446.0


In [25]:
late_sum.to_csv('../Late_embryo/Late_embryo_sum.csv')

In [26]:
concatenated_df.to_csv('../Late_embryo/Late_conformation_conformation_count.csv')

In [27]:
transposed_df.to_csv('../Late_embryo/Late_embryo_grouped_conformation.csv')

In [28]:
def calculate_column_medians(counts_conformation):
    columns_medians = concatenated_df.median()
    columns_median_df = pd.DataFrame(columns_medians, columns=['Median'])
    columns_median_df.reset_index(inplace=True)
    columns_median_df.rename(columns={'index': 'Categories'}, inplace=True)
    return columns_median_df

df_late_median = calculate_column_medians(counts_conformation)
late_embryo_median = df_late_median.drop([0]).reset_index(drop = True)

In [29]:
late_embryo_median

,Categories,Median
0,Counts_of_not_touching,8.0
1,Counts_3_prime_touch_5_prime,2.0
2,Counts_5_prime_touch_PPE,8.0
3,Counts_3_prime_touch_PPE,3.0
4,Counts_5_prime_middle,0.0
5,Counts_PPE_middle,1.0
6,Counts_3_prime_middle,0.0
7,Counts_all_touching,12.0


In [30]:
late_embryo_median.to_csv("../Late_embryo/late_embryo_median_conformation_count.csv")